# 23. Stable Diffusion Baseline Report

This notebook generates a consolidated HTML report for the Stable Diffusion Inpainting baseline on the controlled 50-painting subset.

The report summarizes:

- restoration metadata,
- classical metric behavior,
- LPIPS perceptual-distance behavior,
- CLIP/DINOv2 feature-similarity behavior,
- selected visual diagnostics,
- spatial error-map diagnostics.

The report is intended as a model-level summary before multi-model comparison.

## Report design

The Stable Diffusion report is not used as a final judgment of restoration quality.

It is a structured diagnostic report that combines quantitative metrics and selected visual examples.

The report emphasizes:

- performance differences across damage types,
- performance differences across painting categories,
- strongest and weakest local cases,
- metric disagreement,
- spatial error-map evidence,
- limitations of interpreting generative outputs as faithful restoration.

Because Stable Diffusion is a generative model, visual plausibility is treated separately from measured similarity to the clean reference.

In [1]:
from pathlib import Path
import sys
import base64
from datetime import datetime

import pandas as pd
import yaml
from PIL import Image
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Project root:", PROJECT_ROOT)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval


In [2]:
config_path = PROJECT_ROOT / "config" / "experiment_50_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}")

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

paths_cfg = config["paths"]

processed_metadata_dir = PROJECT_ROOT / paths_cfg["processed_metadata_dir"]
metrics_dir = PROJECT_ROOT / paths_cfg["metrics_dir"]
figures_dir = PROJECT_ROOT / paths_cfg["figures_dir"]
reports_dir = PROJECT_ROOT / paths_cfg.get("reports_dir", "outputs/reports")

metrics_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

model_name = "stable_diffusion_inpainting"
model_display_name = "Stable Diffusion Inpainting"

restored_metadata_path = processed_metadata_dir / "metadata_restored_stable_diffusion.csv"

classical_metrics_path = metrics_dir / "classical_metrics_stable_diffusion_50.csv"
classical_summary_by_mask_type_path = metrics_dir / "classical_metrics_summary_by_mask_type_stable_diffusion_50.csv"
classical_summary_by_category_path = metrics_dir / "classical_metrics_summary_by_category_stable_diffusion_50.csv"
classical_masked_region_summary_path = metrics_dir / "classical_metrics_masked_region_summary_stable_diffusion_50.csv"

lpips_metrics_path = metrics_dir / "lpips_metrics_stable_diffusion_50.csv"
lpips_summary_by_mask_type_path = metrics_dir / "lpips_metrics_summary_by_mask_type_stable_diffusion_50.csv"
lpips_summary_by_category_path = metrics_dir / "lpips_metrics_summary_by_category_stable_diffusion_50.csv"
lpips_mask_bbox_summary_path = metrics_dir / "lpips_metrics_masked_region_summary_stable_diffusion_50.csv"

feature_metrics_path = metrics_dir / "feature_similarity_stable_diffusion_50.csv"
feature_summary_by_mask_type_path = metrics_dir / "feature_similarity_summary_by_mask_type_stable_diffusion_50.csv"
feature_summary_by_category_path = metrics_dir / "feature_similarity_summary_by_category_stable_diffusion_50.csv"
feature_mask_bbox_summary_path = metrics_dir / "feature_similarity_mask_bbox_summary_stable_diffusion_50.csv"

error_map_manifest_path = metrics_dir / "error_map_manifest_stable_diffusion_50.csv"
error_map_visual_cases_path = metrics_dir / "stable_diffusion_error_map_visual_cases_50.csv"

classical_visual_cases_path = metrics_dir / "stable_diffusion_classical_metric_visual_cases_manifest_50.csv"
lpips_visual_cases_path = metrics_dir / "stable_diffusion_lpips_visual_cases_manifest_50.csv"
feature_visual_cases_path = metrics_dir / "stable_diffusion_feature_similarity_visual_cases_manifest_50.csv"

report_dataframe_path = metrics_dir / "stable_diffusion_report_dataframe_50.csv"
report_selected_cases_path = metrics_dir / "stable_diffusion_report_selected_cases_50.csv"
html_report_path = reports_dir / "stable_diffusion_baseline_report_50.html"

print("Stable Diffusion report output:", html_report_path)

Stable Diffusion report output: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\stable_diffusion_baseline_report_50.html


In [3]:
required_input_paths = [
    restored_metadata_path,
    classical_metrics_path,
    classical_summary_by_mask_type_path,
    classical_summary_by_category_path,
    classical_masked_region_summary_path,
    lpips_metrics_path,
    lpips_summary_by_mask_type_path,
    lpips_summary_by_category_path,
    lpips_mask_bbox_summary_path,
    feature_metrics_path,
    feature_summary_by_mask_type_path,
    feature_summary_by_category_path,
    feature_mask_bbox_summary_path,
    error_map_manifest_path,
    error_map_visual_cases_path,
    classical_visual_cases_path,
    lpips_visual_cases_path,
    feature_visual_cases_path,
]

for input_path in required_input_paths:
    if not input_path.exists():
        raise FileNotFoundError(f"Missing required report input: {input_path}")

restored_df = pd.read_csv(restored_metadata_path)

classical_metrics_df = pd.read_csv(classical_metrics_path)
classical_summary_by_mask_type_df = pd.read_csv(classical_summary_by_mask_type_path)
classical_summary_by_category_df = pd.read_csv(classical_summary_by_category_path)
classical_masked_region_summary_df = pd.read_csv(classical_masked_region_summary_path)

lpips_metrics_df = pd.read_csv(lpips_metrics_path)
lpips_summary_by_mask_type_df = pd.read_csv(lpips_summary_by_mask_type_path)
lpips_summary_by_category_df = pd.read_csv(lpips_summary_by_category_path)
lpips_mask_bbox_summary_df = pd.read_csv(lpips_mask_bbox_summary_path)

feature_metrics_df = pd.read_csv(feature_metrics_path)
feature_summary_by_mask_type_df = pd.read_csv(feature_summary_by_mask_type_path)
feature_summary_by_category_df = pd.read_csv(feature_summary_by_category_path)
feature_mask_bbox_summary_df = pd.read_csv(feature_mask_bbox_summary_path)

error_map_manifest_df = pd.read_csv(error_map_manifest_path)
error_map_visual_cases_df = pd.read_csv(error_map_visual_cases_path)

classical_visual_cases_df = pd.read_csv(classical_visual_cases_path)
lpips_visual_cases_df = pd.read_csv(lpips_visual_cases_path)
feature_visual_cases_df = pd.read_csv(feature_visual_cases_path)

print("Restored metadata:", restored_df.shape)
print("Classical metrics:", classical_metrics_df.shape)
print("LPIPS metrics:", lpips_metrics_df.shape)
print("Feature metrics:", feature_metrics_df.shape)
print("Error-map manifest:", error_map_manifest_df.shape)
print("Classical visual cases:", classical_visual_cases_df.shape)
print("LPIPS visual cases:", lpips_visual_cases_df.shape)
print("Feature visual cases:", feature_visual_cases_df.shape)

Restored metadata: (250, 36)
Classical metrics: (900, 30)
LPIPS metrics: (700, 24)
Feature metrics: (700, 28)
Error-map manifest: (53, 29)
Classical visual cases: (14, 36)
LPIPS visual cases: (32, 31)
Feature visual cases: (44, 38)


In [4]:
if len(restored_df) != 250:
    raise ValueError(f"Expected 250 restoration rows, found {len(restored_df)}.")

if len(classical_metrics_df) != 900:
    raise ValueError(f"Expected 900 classical metric rows, found {len(classical_metrics_df)}.")

if len(lpips_metrics_df) != 700:
    raise ValueError(f"Expected 700 LPIPS metric rows, found {len(lpips_metrics_df)}.")

if len(feature_metrics_df) != 700:
    raise ValueError(f"Expected 700 feature-similarity metric rows, found {len(feature_metrics_df)}.")

for name, df in {
    "restored": restored_df,
    "classical": classical_metrics_df,
    "lpips": lpips_metrics_df,
    "feature": feature_metrics_df,
}.items():
    if "status" in df.columns and (df["status"] != "ok").any():
        display(df[df["status"] != "ok"])
        raise ValueError(f"{name} dataframe contains non-ok rows.")

if set(restored_df["model_name"].dropna().unique()) != {model_name}:
    raise ValueError(f"Unexpected restored model names: {restored_df['model_name'].unique()}")

expected_restoration_mask_counts = {
    "loss_large": 50,
    "loss_small": 50,
    "mixed_damage": 50,
    "scratch_thin": 50,
    "zero_control": 50,
}

actual_restoration_mask_counts = restored_df["mask_type"].value_counts().to_dict()

print("Expected restoration mask counts:", expected_restoration_mask_counts)
print("Actual restoration mask counts:", actual_restoration_mask_counts)

for mask_type, expected_count in expected_restoration_mask_counts.items():
    actual_count = actual_restoration_mask_counts.get(mask_type, 0)

    if actual_count != expected_count:
        raise ValueError(
            f"Restoration mask count mismatch for {mask_type!r}: "
            f"expected {expected_count}, found {actual_count}."
        )

expected_classical_regions = {
    "full_image": 250,
    "content_region": 250,
    "masked_region": 200,
    "mask_bbox_crop": 200,
}

expected_feature_like_regions = {
    "full_image": 250,
    "content_region": 250,
    "mask_bbox_crop": 200,
}

actual_classical_regions = classical_metrics_df["evaluation_region"].value_counts().to_dict()
actual_lpips_regions = lpips_metrics_df["evaluation_region"].value_counts().to_dict()
actual_feature_regions = feature_metrics_df["evaluation_region"].value_counts().to_dict()

print("\nClassical regions:", actual_classical_regions)
print("LPIPS regions:", actual_lpips_regions)
print("Feature regions:", actual_feature_regions)

for region, expected_count in expected_classical_regions.items():
    if actual_classical_regions.get(region, 0) != expected_count:
        raise ValueError(
            f"Classical region {region!r}: expected {expected_count}, "
            f"found {actual_classical_regions.get(region, 0)}."
        )

for metric_name, actual_regions in {
    "LPIPS": actual_lpips_regions,
    "feature": actual_feature_regions,
}.items():
    for region, expected_count in expected_feature_like_regions.items():
        if actual_regions.get(region, 0) != expected_count:
            raise ValueError(
                f"{metric_name} region {region!r}: expected {expected_count}, "
                f"found {actual_regions.get(region, 0)}."
            )

print("\nStable Diffusion report input gates passed.")

Expected restoration mask counts: {'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50, 'zero_control': 50}
Actual restoration mask counts: {'loss_large': 50, 'loss_small': 50, 'mixed_damage': 50, 'scratch_thin': 50, 'zero_control': 50}

Classical regions: {'full_image': 250, 'content_region': 250, 'masked_region': 200, 'mask_bbox_crop': 200}
LPIPS regions: {'full_image': 250, 'content_region': 250, 'mask_bbox_crop': 200}
Feature regions: {'full_image': 250, 'content_region': 250, 'mask_bbox_crop': 200}

Stable Diffusion report input gates passed.


In [5]:
def find_single_column(df: pd.DataFrame, required_terms: list[str]) -> str:
    matches = []

    for column in df.columns:
        column_lower = column.lower()

        if all(term.lower() in column_lower for term in required_terms):
            matches.append(column)

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one column matching terms {required_terms}, "
            f"found {len(matches)}: {matches}"
        )

    return matches[0]


def resolve_project_path(path_value: str | Path) -> Path:
    path = Path(str(path_value))

    if path.is_absolute():
        return path

    return PROJECT_ROOT / path


def dataframe_to_html_table(
    df: pd.DataFrame,
    *,
    max_rows: int = 20,
    float_precision: int = 4,
) -> str:
    display_df = df.head(max_rows).copy()

    for column in display_df.select_dtypes(include=["float", "float64", "float32"]).columns:
        display_df[column] = display_df[column].round(float_precision)

    return display_df.to_html(index=False, escape=True, classes="data-table")


def image_to_base64_data_uri(path_value: str | Path, *, max_width: int = 1400) -> str:
    path = resolve_project_path(path_value)

    if not path.exists():
        raise FileNotFoundError(f"Missing image for report embedding: {path}")

    image = Image.open(path).convert("RGB")

    if image.width > max_width:
        scale = max_width / image.width
        new_size = (max_width, int(image.height * scale))
        image = image.resize(new_size)

    from io import BytesIO

    buffer = BytesIO()
    image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return f"data:image/png;base64,{encoded}"


print("Report helper functions ready.")

Report helper functions ready.


In [6]:
classical_mse_improvement_column = find_single_column(
    classical_metrics_df,
    ["mse", "improvement"],
)

classical_psnr_improvement_column = find_single_column(
    classical_metrics_df,
    ["psnr", "improvement"],
)

classical_ssim_improvement_column = find_single_column(
    classical_metrics_df,
    ["ssim", "improvement"],
)

lpips_improvement_column = find_single_column(
    lpips_metrics_df,
    ["lpips", "improvement"],
)

clip_improvement_column = find_single_column(
    feature_metrics_df,
    ["clip", "improvement"],
)

dinov2_improvement_column = find_single_column(
    feature_metrics_df,
    ["dinov2", "improvement"],
)

print("Detected report metric columns:")
print("Classical MSE improvement:", classical_mse_improvement_column)
print("Classical PSNR improvement:", classical_psnr_improvement_column)
print("Classical SSIM improvement:", classical_ssim_improvement_column)
print("LPIPS improvement:", lpips_improvement_column)
print("CLIP improvement:", clip_improvement_column)
print("DINOv2 improvement:", dinov2_improvement_column)

Detected report metric columns:
Classical MSE improvement: mse_improvement
Classical PSNR improvement: psnr_improvement
Classical SSIM improvement: ssim_improvement
LPIPS improvement: lpips_improvement
CLIP improvement: clip_similarity_improvement
DINOv2 improvement: dinov2_similarity_improvement


In [7]:
classical_local_df = classical_metrics_df[
    classical_metrics_df["evaluation_region"] == "masked_region"
].copy()

lpips_local_df = lpips_metrics_df[
    lpips_metrics_df["evaluation_region"] == "mask_bbox_crop"
].copy()

feature_local_df = feature_metrics_df[
    feature_metrics_df["evaluation_region"] == "mask_bbox_crop"
].copy()

if len(classical_local_df) != 200:
    raise ValueError(
        f"Expected 200 classical local rows, found {len(classical_local_df)}."
    )

if len(lpips_local_df) != 200:
    raise ValueError(
        f"Expected 200 LPIPS local rows, found {len(lpips_local_df)}."
    )

if len(feature_local_df) != 200:
    raise ValueError(
        f"Expected 200 feature local rows, found {len(feature_local_df)}."
    )

report_base_columns = [
    "case_id",
    "painting_id",
    "category",
    "title",
    "mask_type",
]

classical_report_df = classical_local_df[
    report_base_columns
    + [
        classical_mse_improvement_column,
        classical_psnr_improvement_column,
        classical_ssim_improvement_column,
    ]
].copy()

lpips_report_df = lpips_local_df[
    [
        "case_id",
        lpips_improvement_column,
    ]
].copy()

feature_report_df = feature_local_df[
    [
        "case_id",
        clip_improvement_column,
        dinov2_improvement_column,
    ]
].copy()

report_df = (
    classical_report_df
    .merge(lpips_report_df, on="case_id", how="inner", validate="one_to_one")
    .merge(feature_report_df, on="case_id", how="inner", validate="one_to_one")
)

if len(report_df) != 200:
    raise ValueError(
        f"Expected 200 local report rows, found {len(report_df)}."
    )

report_df["mse_improved"] = report_df[classical_mse_improvement_column] > 0
report_df["psnr_improved"] = report_df[classical_psnr_improvement_column] > 0
report_df["ssim_improved"] = report_df[classical_ssim_improvement_column] > 0
report_df["lpips_improved"] = report_df[lpips_improvement_column] > 0
report_df["clip_improved"] = report_df[clip_improvement_column] > 0
report_df["dinov2_improved"] = report_df[dinov2_improvement_column] > 0

improvement_flag_columns = [
    "mse_improved",
    "psnr_improved",
    "ssim_improved",
    "lpips_improved",
    "clip_improved",
    "dinov2_improved",
]

report_df["num_metrics_improved"] = report_df[improvement_flag_columns].sum(axis=1)
report_df["num_metrics_not_improved"] = len(improvement_flag_columns) - report_df["num_metrics_improved"]

report_df["all_metrics_improved"] = (
    report_df["num_metrics_improved"] == len(improvement_flag_columns)
)

report_df["mixed_metric_outcome"] = report_df["num_metrics_improved"].between(
    1,
    len(improvement_flag_columns) - 1,
)

report_df["all_metrics_not_improved"] = report_df["num_metrics_improved"] == 0

report_df.to_csv(report_dataframe_path, index=False)

print("Stable Diffusion report dataframe shape:", report_df.shape)
print("Saved report dataframe:", report_dataframe_path)

display(report_df.head())

Stable Diffusion report dataframe shape: (200, 22)
Saved report dataframe: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\stable_diffusion_report_dataframe_50.csv


,case_id,painting_id,category,title,mask_type,mse_improvement,psnr_improvement,ssim_improvement,lpips_improvement,clip_similarity_improvement,...,psnr_improved,ssim_improved,lpips_improved,clip_improved,dinov2_improved,num_metrics_improved,num_metrics_not_improved,all_metrics_improved,mixed_metric_outcome,all_metrics_not_improved
0,p001_loss_large,p001,portrait_figure,Juan de Pareja,loss_large,48133.030640,17.532740,NaN,0.391132,0.119137,...,True,False,True,True,True,5,1,False,True,False
1,p001_loss_small,p001,portrait_figure,Juan de Pareja,loss_small,46912.756538,27.543226,NaN,0.306002,0.071142,...,True,False,True,True,True,5,1,False,True,False
2,p001_mixed_damage,p001,portrait_figure,Juan de Pareja,mixed_damage,49826.728882,17.444243,NaN,0.244464,0.060504,...,True,False,True,True,True,5,1,False,True,False
3,p001_scratch_thin,p001,portrait_figure,Juan de Pareja,scratch_thin,42301.111328,8.929056,NaN,0.161720,0.043441,...,True,False,True,True,False,4,2,False,True,False
4,p002_loss_large,p002,portrait_figure,Madame X (Madame Pierre Gautreau),loss_large,50524.246399,18.609613,NaN,0.407476,0.222473,...,True,False,True,True,True,5,1,False,True,False


In [8]:
metric_overview_df = pd.DataFrame(
    [
        {
            "metric_layer": "restoration_outputs",
            "rows": len(restored_df),
            "local_region_used_in_report": "non_zero_cases",
        },
        {
            "metric_layer": "classical_metrics",
            "rows": len(classical_metrics_df),
            "local_region_used_in_report": "masked_region",
        },
        {
            "metric_layer": "lpips_metrics",
            "rows": len(lpips_metrics_df),
            "local_region_used_in_report": "mask_bbox_crop",
        },
        {
            "metric_layer": "feature_similarity_metrics",
            "rows": len(feature_metrics_df),
            "local_region_used_in_report": "mask_bbox_crop",
        },
        {
            "metric_layer": "error_map_diagnostics",
            "rows": len(error_map_manifest_df),
            "local_region_used_in_report": "selected_cases",
        },
    ]
)

local_summary_by_mask_type_df = (
    report_df
    .groupby("mask_type", dropna=False)
    .agg(
        cases=("case_id", "count"),
        mean_mse_improvement=(classical_mse_improvement_column, "mean"),
        median_mse_improvement=(classical_mse_improvement_column, "median"),
        mean_lpips_improvement=(lpips_improvement_column, "mean"),
        median_lpips_improvement=(lpips_improvement_column, "median"),
        mean_clip_improvement=(clip_improvement_column, "mean"),
        median_clip_improvement=(clip_improvement_column, "median"),
        mean_dinov2_improvement=(dinov2_improvement_column, "mean"),
        median_dinov2_improvement=(dinov2_improvement_column, "median"),
        mean_num_metrics_improved=("num_metrics_improved", "mean"),
    )
    .reset_index()
    .sort_values("mask_type")
)

local_summary_by_category_df = (
    report_df
    .groupby("category", dropna=False)
    .agg(
        cases=("case_id", "count"),
        mean_mse_improvement=(classical_mse_improvement_column, "mean"),
        median_mse_improvement=(classical_mse_improvement_column, "median"),
        mean_lpips_improvement=(lpips_improvement_column, "mean"),
        median_lpips_improvement=(lpips_improvement_column, "median"),
        mean_clip_improvement=(clip_improvement_column, "mean"),
        median_clip_improvement=(clip_improvement_column, "median"),
        mean_dinov2_improvement=(dinov2_improvement_column, "mean"),
        median_dinov2_improvement=(dinov2_improvement_column, "median"),
        mean_num_metrics_improved=("num_metrics_improved", "mean"),
    )
    .reset_index()
    .sort_values("category")
)

metric_outcome_summary_df = (
    report_df
    .groupby("num_metrics_improved", dropna=False)
    .agg(cases=("case_id", "count"))
    .reset_index()
    .sort_values("num_metrics_improved")
)

print("Metric overview:")
display(metric_overview_df)

print("\nLocal summary by mask type:")
display(local_summary_by_mask_type_df)

print("\nLocal summary by category:")
display(local_summary_by_category_df)

print("\nMetric outcome summary:")
display(metric_outcome_summary_df)

Metric overview:


,metric_layer,rows,local_region_used_in_report
0,restoration_outputs,250,non_zero_cases
1,classical_metrics,900,masked_region
2,lpips_metrics,700,mask_bbox_crop
3,feature_similarity_metrics,700,mask_bbox_crop
4,error_map_diagnostics,53,selected_cases



Local summary by mask type:


,mask_type,cases,mean_mse_improvement,median_mse_improvement,mean_lpips_improvement,median_lpips_improvement,mean_clip_improvement,median_clip_improvement,mean_dinov2_improvement,median_dinov2_improvement,mean_num_metrics_improved
0,loss_large,50,23124.429134,23581.619751,0.221407,0.209235,0.106688,0.111895,0.039577,0.044629,4.58
1,loss_small,50,25636.699247,24373.020355,0.108700,0.098342,0.073849,0.067969,0.020351,0.004739,4.60
2,mixed_damage,50,25735.033149,24461.882599,0.188334,0.191327,0.104924,0.107086,0.110759,0.082177,4.86
3,scratch_thin,50,25670.202737,26801.796753,0.131230,0.147154,0.048619,0.046759,0.018038,0.003239,4.30



Local summary by category:


,category,cases,mean_mse_improvement,median_mse_improvement,mean_lpips_improvement,median_lpips_improvement,mean_clip_improvement,median_clip_improvement,mean_dinov2_improvement,median_dinov2_improvement,mean_num_metrics_improved
0,abstraction_surrealism,40,17876.773131,20654.544434,0.105644,0.105972,0.063532,0.047703,0.014449,0.005044,4.325
1,architecture_structured,40,24537.514644,24271.189636,0.188694,0.182781,0.095640,0.086500,0.008588,-0.002673,4.450
2,high_texture_brushwork,40,25769.689114,25474.678040,0.173777,0.173329,0.099730,0.091653,0.103444,0.089875,4.775
3,landscape_natural,40,20384.583746,19171.336365,0.136462,0.133128,0.069364,0.067814,0.045523,0.030710,4.550
4,portrait_figure,40,36639.394696,35595.753143,0.207513,0.196408,0.089334,0.087660,0.063903,0.050597,4.825



Metric outcome summary:


,num_metrics_improved,cases
0,0,1
1,1,1
2,2,1
3,3,6
4,4,59
5,5,132


In [9]:
best_report_cases_df = (
    report_df
    .sort_values("num_metrics_improved", ascending=False)
    .head(10)
    .copy()
)

best_report_cases_df["report_selection_reason"] = "highest_number_of_improved_metrics"

worst_report_cases_df = (
    report_df
    .sort_values("num_metrics_improved", ascending=True)
    .head(10)
    .copy()
)

worst_report_cases_df["report_selection_reason"] = "lowest_number_of_improved_metrics"

mixed_report_cases_df = (
    report_df[
        report_df["mixed_metric_outcome"]
    ]
    .sort_values(
        [
            "num_metrics_improved",
            classical_mse_improvement_column,
        ],
        ascending=[True, True],
    )
    .head(10)
    .copy()
)

mixed_report_cases_df["report_selection_reason"] = "mixed_metric_outcome"

category_mask_report_cases_df = (
    report_df
    .sort_values("num_metrics_improved", ascending=False)
    .groupby(["category", "mask_type"], dropna=False)
    .head(1)
    .copy()
)

category_mask_report_cases_df["report_selection_reason"] = "category_mask_representative"

report_cases_raw_df = pd.concat(
    [
        best_report_cases_df,
        worst_report_cases_df,
        mixed_report_cases_df,
        category_mask_report_cases_df,
    ],
    ignore_index=True,
)

reason_df = (
    report_cases_raw_df
    .groupby("case_id", dropna=False)
    .agg(
        report_selection_reason=(
            "report_selection_reason",
            lambda values: "; ".join(sorted(set(values))),
        )
    )
    .reset_index()
)

report_case_base_columns = [
    column for column in report_cases_raw_df.columns
    if column != "report_selection_reason"
]

report_selected_cases_df = (
    report_cases_raw_df[report_case_base_columns]
    .drop_duplicates(subset=["case_id"])
    .merge(reason_df, on="case_id", how="left", validate="one_to_one")
    .sort_values(["category", "mask_type", "case_id"])
    .reset_index(drop=True)
)

report_selected_cases_df.to_csv(report_selected_cases_path, index=False)

print("Report selected cases:", len(report_selected_cases_df))
print("Saved report selected cases:", report_selected_cases_path)

display(
    report_selected_cases_df[
        [
            "case_id",
            "category",
            "title",
            "mask_type",
            "report_selection_reason",
            "num_metrics_improved",
            classical_mse_improvement_column,
            lpips_improvement_column,
            clip_improvement_column,
            dinov2_improvement_column,
        ]
    ]
)

Report selected cases: 37
Saved report selected cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\stable_diffusion_report_selected_cases_50.csv


,case_id,category,title,mask_type,report_selection_reason,num_metrics_improved,mse_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement
0,p031_loss_large,abstraction_surrealism,Improvisation No. 30 (Cannons),loss_large,category_mask_representative; highest_number_o...,5,15065.325928,0.155152,0.148135,0.063570
1,p032_loss_large,abstraction_surrealism,Composition (No. 1) Gray-Red,loss_large,highest_number_of_improved_metrics,5,7215.788330,0.268420,0.055404,0.176233
2,p036_loss_large,abstraction_surrealism,"Lozenge Composition with Yellow, Black, Blue, ...",loss_large,lowest_number_of_improved_metrics; mixed_metri...,2,-1520.606445,0.137071,0.016907,-0.031298
3,p032_loss_small,abstraction_surrealism,Composition (No. 1) Gray-Red,loss_small,category_mask_representative; highest_number_o...,5,8251.094635,0.113354,0.092082,0.034382
4,p033_loss_small,abstraction_surrealism,Movement,loss_small,highest_number_of_improved_metrics,5,21045.137207,0.052338,0.023541,0.005874
5,p036_loss_small,abstraction_surrealism,"Lozenge Composition with Yellow, Black, Blue, ...",loss_small,mixed_metric_outcome,4,3229.334770,0.058343,0.061914,-0.008986
6,p038_loss_small,abstraction_surrealism,Mahana no atua (Day of the God),loss_small,lowest_number_of_improved_metrics; mixed_metri...,3,23013.376465,0.025004,-0.013936,-0.040801
7,p031_mixed_damage,abstraction_surrealism,Improvisation No. 30 (Cannons),mixed_damage,category_mask_representative; highest_number_o...,5,16620.918945,0.115954,0.031995,0.081709
8,p032_mixed_damage,abstraction_surrealism,Composition (No. 1) Gray-Red,mixed_damage,highest_number_of_improved_metrics,5,1885.202820,0.036408,0.040001,0.052295
9,p033_mixed_damage,abstraction_surrealism,Movement,mixed_damage,highest_number_of_improved_metrics,5,25415.051758,0.126359,0.032656,0.063161


In [10]:
def find_figure_path_column(df: pd.DataFrame) -> str:
    figure_path_columns = [
        column
        for column in df.columns
        if "figure" in column.lower() and "path" in column.lower()
    ]

    if len(figure_path_columns) != 1:
        raise ValueError(
            f"Expected exactly one figure path column, found {len(figure_path_columns)}: "
            f"{figure_path_columns}"
        )

    return figure_path_columns[0]


classical_figure_column = find_figure_path_column(classical_visual_cases_df)
lpips_figure_column = find_figure_path_column(lpips_visual_cases_df)
feature_figure_column = find_figure_path_column(feature_visual_cases_df)
error_map_figure_column = find_figure_path_column(error_map_manifest_df)

print("Classical figure column:", classical_figure_column)
print("LPIPS figure column:", lpips_figure_column)
print("Feature figure column:", feature_figure_column)
print("Error-map figure column:", error_map_figure_column)

classical_report_visual_df = (
    classical_visual_cases_df
    .sort_values(["selection_reason", "category", "mask_type", "case_id"])
    .groupby("selection_reason", dropna=False)
    .head(1)
    .copy()
)

lpips_report_visual_df = (
    lpips_visual_cases_df
    .sort_values(["selection_reason", "category", "mask_type", "case_id"])
    .groupby("selection_reason", dropna=False)
    .head(1)
    .copy()
)

feature_report_visual_df = (
    feature_visual_cases_df
    .sort_values(["selection_reason", "category", "mask_type", "case_id"])
    .groupby("selection_reason", dropna=False)
    .head(1)
    .copy()
)

error_map_report_visual_df = (
    error_map_manifest_df
    .merge(
        error_map_visual_cases_df[
            ["case_id", "selection_reason", "mse_improvement", "psnr_improvement", "ssim_improvement"]
        ],
        on="case_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(["selection_reason", "category", "mask_type", "case_id"])
    .groupby("selection_reason", dropna=False)
    .head(1)
    .copy()
)

print("Classical report visual figures:", len(classical_report_visual_df))
print("LPIPS report visual figures:", len(lpips_report_visual_df))
print("Feature report visual figures:", len(feature_report_visual_df))
print("Error-map report visual figures:", len(error_map_report_visual_df))

display(
    classical_report_visual_df[
        ["case_id", "category", "mask_type", "selection_reason", classical_figure_column]
    ]
)

Classical figure column: visual_figure_path
LPIPS figure column: visual_figure_path
Feature figure column: visual_figure_path
Error-map figure column: figure_path
Classical report visual figures: 4
LPIPS report visual figures: 5
Feature report visual figures: 14
Error-map report visual figures: 7


,case_id,category,mask_type,selection_reason,visual_figure_path
1,p037_loss_large,abstraction_surrealism,loss_large,Strong category example by masked-region MSE i...,D:\Masters\FH\Thesis\painting-restoration-eval...
13,p006_mixed_damage,portrait_figure,mixed_damage,Strong category example by masked-region MSE i...,D:\Masters\FH\Thesis\painting-restoration-eval...
9,p001_loss_large,portrait_figure,loss_large,Strongest masked-region MSE improvement,D:\Masters\FH\Thesis\painting-restoration-eval...
0,p036_loss_large,abstraction_surrealism,loss_large,Weakest masked-region MSE improvement,D:\Masters\FH\Thesis\painting-restoration-eval...


In [11]:
def render_metric_card(title: str, value: str, subtitle: str = "") -> str:
    return f"""
    <div class="metric-card">
        <div class="metric-title">{title}</div>
        <div class="metric-value">{value}</div>
        <div class="metric-subtitle">{subtitle}</div>
    </div>
    """


def render_visual_gallery(
    df: pd.DataFrame,
    *,
    figure_column: str,
    section_title: str,
    max_items: int = 6,
) -> str:
    gallery_items = []

    for _, row in df.head(max_items).iterrows():
        figure_path = row[figure_column]
        image_uri = image_to_base64_data_uri(figure_path, max_width=1500)

        case_id = row.get("case_id", "")
        category = row.get("category", "")
        mask_type = row.get("mask_type", "")
        reason = row.get("selection_reason", row.get("report_selection_reason", ""))

        gallery_items.append(
            f"""
            <div class="visual-item">
                <h4>{case_id} | {category} | {mask_type}</h4>
                <p class="reason">{reason}</p>
                <img src="{image_uri}" alt="{case_id}">
            </div>
            """
        )

    return f"""
    <section>
        <h2>{section_title}</h2>
        <div class="visual-gallery">
            {''.join(gallery_items)}
        </div>
    </section>
    """


def render_report_case_table(df: pd.DataFrame, *, max_rows: int = 30) -> str:
    columns = [
        "case_id",
        "category",
        "title",
        "mask_type",
        "report_selection_reason",
        "num_metrics_improved",
        classical_mse_improvement_column,
        lpips_improvement_column,
        clip_improvement_column,
        dinov2_improvement_column,
    ]

    available_columns = [
        column for column in columns
        if column in df.columns
    ]

    return dataframe_to_html_table(
        df[available_columns],
        max_rows=max_rows,
        float_precision=5,
    )


print("HTML rendering helpers ready.")

HTML rendering helpers ready.


In [12]:
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

non_zero_rows = restored_df[restored_df["mask_type"] != "zero_control"]
zero_control_rows = restored_df[restored_df["mask_type"] == "zero_control"]

metric_cards_html = "".join(
    [
        render_metric_card(
            "Restoration cases",
            str(len(restored_df)),
            "50 paintings × 5 mask types",
        ),
        render_metric_card(
            "Model inference cases",
            str(len(non_zero_rows)),
            "Non-zero damage masks",
        ),
        render_metric_card(
            "Zero controls",
            str(len(zero_control_rows)),
            "Copied directly by design",
        ),
        render_metric_card(
            "Classical metric rows",
            str(len(classical_metrics_df)),
            "Includes masked-region rows",
        ),
        render_metric_card(
            "LPIPS rows",
            str(len(lpips_metrics_df)),
            "Uses mask_bbox_crop as local proxy",
        ),
        render_metric_card(
            "Feature rows",
            str(len(feature_metrics_df)),
            "CLIP and DINOv2 similarity",
        ),
        render_metric_card(
            "Local report cases",
            str(len(report_df)),
            "Non-zero damage cases",
        ),
        render_metric_card(
            "Selected report cases",
            str(len(report_selected_cases_df)),
            "Fixed diagnostic policy",
        ),
    ]
)

classical_tables_html = f"""
<section>
    <h2>Classical metric summaries</h2>
    <h3>By mask type</h3>
    {dataframe_to_html_table(classical_summary_by_mask_type_df, max_rows=20)}
    <h3>By category</h3>
    {dataframe_to_html_table(classical_summary_by_category_df, max_rows=20)}
    <h3>Masked-region summary</h3>
    {dataframe_to_html_table(classical_masked_region_summary_df, max_rows=20)}
</section>
"""

lpips_tables_html = f"""
<section>
    <h2>LPIPS perceptual-distance summaries</h2>
    <p>Lower LPIPS means lower perceptual distance. Positive improvement means the restored output is closer to the clean reference than the damaged input.</p>
    <h3>By mask type</h3>
    {dataframe_to_html_table(lpips_summary_by_mask_type_df, max_rows=20)}
    <h3>By category</h3>
    {dataframe_to_html_table(lpips_summary_by_category_df, max_rows=20)}
    <h3>Mask-bounding-box summary</h3>
    {dataframe_to_html_table(lpips_mask_bbox_summary_df, max_rows=20)}
</section>
"""

feature_tables_html = f"""
<section>
    <h2>CLIP/DINOv2 feature-similarity summaries</h2>
    <p>Higher cosine similarity means higher feature-space similarity to the clean reference. Positive improvement means the restored output is closer to the clean reference than the damaged input.</p>
    <h3>By mask type</h3>
    {dataframe_to_html_table(feature_summary_by_mask_type_df, max_rows=20)}
    <h3>By category</h3>
    {dataframe_to_html_table(feature_summary_by_category_df, max_rows=20)}
    <h3>Mask-bounding-box summary</h3>
    {dataframe_to_html_table(feature_mask_bbox_summary_df, max_rows=20)}
</section>
"""

local_report_tables_html = f"""
<section>
    <h2>Local report summary</h2>
    <p>This table combines local metrics for the 200 non-zero damage cases. Classical metrics use the sparse masked region. LPIPS and feature-space metrics use the mask-bounding-box crop.</p>
    <h3>Metric layer overview</h3>
    {dataframe_to_html_table(metric_overview_df, max_rows=20)}
    <h3>Local summary by mask type</h3>
    {dataframe_to_html_table(local_summary_by_mask_type_df, max_rows=20)}
    <h3>Local summary by category</h3>
    {dataframe_to_html_table(local_summary_by_category_df, max_rows=20)}
    <h3>Number of improved metrics per case</h3>
    {dataframe_to_html_table(metric_outcome_summary_df, max_rows=20)}
</section>
"""

selected_cases_html = f"""
<section>
    <h2>Selected report cases</h2>
    <p>Cases were selected using a fixed policy: highest number of improved metrics, lowest number of improved metrics, mixed metric outcomes, and category/mask representatives.</p>
    {render_report_case_table(report_selected_cases_df, max_rows=40)}
</section>
"""

visual_sections_html = "\n".join(
    [
        render_visual_gallery(
            classical_report_visual_df,
            figure_column=classical_figure_column,
            section_title="Classical metric diagnostic examples",
            max_items=6,
        ),
        render_visual_gallery(
            lpips_report_visual_df,
            figure_column=lpips_figure_column,
            section_title="LPIPS diagnostic examples",
            max_items=6,
        ),
        render_visual_gallery(
            feature_report_visual_df,
            figure_column=feature_figure_column,
            section_title="Feature-similarity diagnostic examples",
            max_items=6,
        ),
        render_visual_gallery(
            error_map_report_visual_df,
            figure_column=error_map_figure_column,
            section_title="Spatial error-map diagnostic examples",
            max_items=6,
        ),
    ]
)

html_report = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>{model_display_name} Baseline Report</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 32px;
    background: #f7f7f7;
    color: #222;
}}
h1, h2, h3 {{
    color: #111;
}}
section {{
    background: white;
    padding: 20px;
    margin: 24px 0;
    border-radius: 10px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.12);
}}
.metric-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));
    gap: 14px;
    margin: 20px 0;
}}
.metric-card {{
    background: #ffffff;
    border-left: 5px solid #444;
    padding: 14px;
    border-radius: 8px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.12);
}}
.metric-title {{
    font-size: 13px;
    color: #666;
    text-transform: uppercase;
}}
.metric-value {{
    font-size: 28px;
    font-weight: bold;
    margin: 8px 0;
}}
.metric-subtitle {{
    font-size: 13px;
    color: #777;
}}
.data-table {{
    border-collapse: collapse;
    width: 100%;
    margin: 12px 0 24px 0;
    font-size: 13px;
}}
.data-table th, .data-table td {{
    border: 1px solid #ddd;
    padding: 7px;
    text-align: left;
}}
.data-table th {{
    background: #efefef;
}}
.visual-gallery {{
    display: flex;
    flex-direction: column;
    gap: 24px;
}}
.visual-item {{
    border-top: 1px solid #ddd;
    padding-top: 16px;
}}
.visual-item img {{
    max-width: 100%;
    height: auto;
    border: 1px solid #ccc;
}}
.reason {{
    color: #555;
    font-size: 13px;
}}
.warning {{
    background: #fff7e6;
    border-left: 5px solid #cc8800;
    padding: 14px;
    border-radius: 8px;
}}
</style>
</head>
<body>

<h1>{model_display_name} Baseline Report</h1>
<p><strong>Generated:</strong> {generated_at}</p>
<p><strong>Model name:</strong> {model_name}</p>

<section>
    <h2>Report purpose</h2>
    <p>This report consolidates the Stable Diffusion Inpainting evaluation results for the controlled 50-painting subset.</p>
    <p>The report combines restoration metadata, classical metrics, LPIPS perceptual-distance metrics, CLIP/DINOv2 feature-similarity metrics, selected visual diagnostics, and spatial error-map examples.</p>
    <div class="warning">
        Stable Diffusion is a generative model. Visual plausibility is not interpreted as conservation or art-historical faithfulness. The results are diagnostic signals for the evaluation framework.
    </div>
</section>

<section>
    <h2>Overview</h2>
    <div class="metric-grid">
        {metric_cards_html}
    </div>
</section>

{classical_tables_html}
{lpips_tables_html}
{feature_tables_html}
{local_report_tables_html}
{selected_cases_html}
{visual_sections_html}

<section>
    <h2>Interpretation notes</h2>
    <ul>
        <li>Classical masked-region metrics directly evaluate synthetically damaged pixels.</li>
        <li>LPIPS and feature-similarity metrics use mask-bounding-box crops as local image-like proxies.</li>
        <li>Positive improvement means the restored output is closer to the clean reference than the damaged input for the corresponding metric.</li>
        <li>Metric disagreement is expected for generative models and is useful diagnostic evidence.</li>
        <li>Stable Diffusion outputs may be visually coherent but still inconsistent with the clean reference.</li>
    </ul>
</section>

</body>
</html>
"""

html_report_path.write_text(html_report, encoding="utf-8")

print("Saved Stable Diffusion HTML report:")
print(html_report_path)

Saved Stable Diffusion HTML report:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\stable_diffusion_baseline_report_50.html


In [14]:
expected_report_files = [
    report_dataframe_path,
    report_selected_cases_path,
    html_report_path,
]

for output_file in expected_report_files:
    if not output_file.exists():
        raise FileNotFoundError(
            f"Missing expected Stable Diffusion report output: {output_file}"
        )

saved_report_df = pd.read_csv(report_dataframe_path)
saved_report_selected_cases_df = pd.read_csv(report_selected_cases_path)

if len(saved_report_df) != 200:
    raise ValueError(
        f"Expected 200 local report rows, found {len(saved_report_df)}."
    )

if len(saved_report_selected_cases_df) == 0:
    raise ValueError("Saved Stable Diffusion selected report cases file is empty.")

required_report_columns = [
    "case_id",
    "painting_id",
    "category",
    "title",
    "mask_type",
    classical_mse_improvement_column,
    classical_psnr_improvement_column,
    classical_ssim_improvement_column,
    lpips_improvement_column,
    clip_improvement_column,
    dinov2_improvement_column,
    "num_metrics_improved",
    "mixed_metric_outcome",
]

missing_report_columns = [
    column for column in required_report_columns
    if column not in saved_report_df.columns
]

if missing_report_columns:
    raise ValueError(
        f"Saved Stable Diffusion report dataframe missing columns: {missing_report_columns}"
    )

required_selected_case_columns = [
    "case_id",
    "painting_id",
    "category",
    "title",
    "mask_type",
    "report_selection_reason",
    "num_metrics_improved",
]

missing_selected_case_columns = [
    column for column in required_selected_case_columns
    if column not in saved_report_selected_cases_df.columns
]

if missing_selected_case_columns:
    raise ValueError(
        f"Saved Stable Diffusion selected cases missing columns: "
        f"{missing_selected_case_columns}"
    )

expected_selection_reasons = [
    "highest_number_of_improved_metrics",
    "lowest_number_of_improved_metrics",
    "mixed_metric_outcome",
    "category_mask_representative",
]

selection_reason_text = " ".join(
    saved_report_selected_cases_df["report_selection_reason"]
    .fillna("")
    .astype(str)
    .tolist()
)

missing_selection_reasons = [
    reason
    for reason in expected_selection_reasons
    if reason not in selection_reason_text
]

if missing_selection_reasons:
    raise ValueError(
        f"Missing expected report selection reasons: {missing_selection_reasons}"
    )

html_content = html_report_path.read_text(encoding="utf-8")

required_html_phrases = [
    "Stable Diffusion Inpainting Baseline Report",
    "Classical metric summaries",
    "LPIPS perceptual-distance summaries",
    "CLIP/DINOv2 feature-similarity summaries",
    "Spatial error-map diagnostic examples",
    "Visual plausibility is not interpreted as conservation",
]

missing_html_phrases = [
    phrase
    for phrase in required_html_phrases
    if phrase not in html_content
]

if missing_html_phrases:
    raise ValueError(
        f"HTML report missing expected phrases: {missing_html_phrases}"
    )

if len(html_content) < 10_000:
    raise ValueError(
        f"HTML report seems too small: {len(html_content)} characters."
    )

print("Saved Stable Diffusion report dataframe rows:", len(saved_report_df))
print("Saved Stable Diffusion selected report cases:", len(saved_report_selected_cases_df))
print("HTML report path:", html_report_path)
print("HTML report characters:", len(html_content))
print("Final Stable Diffusion report output gates passed.")

Saved Stable Diffusion report dataframe rows: 200
Saved Stable Diffusion selected report cases: 37
HTML report path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\stable_diffusion_baseline_report_50.html
HTML report characters: 10138819
Final Stable Diffusion report output gates passed.


## Notebook 23 summary

This notebook generated a consolidated Stable Diffusion Inpainting baseline report for the controlled 50-painting subset.

The report combines:

- restoration metadata,
- classical metric summaries,
- LPIPS perceptual-distance summaries,
- CLIP/DINOv2 feature-similarity summaries,
- local metric outcome summaries,
- selected report cases,
- classical metric visual diagnostics,
- LPIPS visual diagnostics,
- feature-similarity visual diagnostics,
- spatial error-map diagnostics.

Main outputs:

- `outputs/reports/stable_diffusion_baseline_report_50.html`
- `outputs/metrics/stable_diffusion_report_dataframe_50.csv`
- `outputs/metrics/stable_diffusion_report_selected_cases_50.csv`

The local report dataframe contains 200 non-zero damage cases.

Classical local metrics use the sparse `masked_region`.

LPIPS and CLIP/DINOv2 local metrics use `mask_bbox_crop`, because these metrics require image-like spatial inputs.

The selected report cases follow a fixed diagnostic policy:

- highest number of improved metrics,
- lowest number of improved metrics,
- mixed metric outcomes,
- category/mask representatives.

This stage completes the Stable Diffusion model-level report and prepares the project for multi-model comparison.